# 2.2. Подбор и изучение источников данных

## Цель этапа

На данном этапе формируется эмпирическая база исследования
«Исследование данных игровых платформ для выявления паттернов
популярности и предпочтений игроков».

В рамках этапа необходимо:

1. Подобрать открытые источники данных.
2. Загрузить исходные данные и сохранить их в каталоге `raw/`.
3. Зафиксировать происхождение, формат и структуру каждого источника.
4. Изучить доступные характеристики объектов исследования.
5. Выделить числовые и категориальные показатели, необходимые
   для последующего анализа.

Данные на этом этапе сохраняются в исходном виде.
Очистка, преобразование и объединение выполняются в ноутбуке `2_3`.

Все пути задаются относительно корня проекта, поэтому ноутбук
можно запускать на разных устройствах после клонирования репозитория.

In [1]:
from pathlib import Path
import os
import sys
import json
import shutil
import subprocess
from datetime import datetime, timezone

import pandas as pd
import requests
from dotenv import load_dotenv
from IPython.display import display
import kagglehub

ROOT = Path.cwd()

# Если ноутбук запущен из каталога notebooks
if ROOT.name == "notebooks":
    ROOT = ROOT.parent

RAW = ROOT / "raw"

for folder in [
    "steam",
    "rawg",
    "twitch",
    "youtube",
    "habr",
    "rosstat",
]:
    (RAW / folder).mkdir(parents=True, exist_ok=True)

load_dotenv(ROOT / ".env")

print("Корень проекта:", ROOT)
print("Каталог исходных данных:", RAW)

Корень проекта: C:\Users\Daniil\Documents\GitHub\gaming-data-analysis
Каталог исходных данных: C:\Users\Daniil\Documents\GitHub\gaming-data-analysis\raw


## 1. Подготовка доступа к источникам

Для загрузки Kaggle-датасетов необходима авторизация Kaggle API.
На каждом устройстве она настраивается отдельно.

Также для получения данных YouTube используется API-ключ
YouTube Data API v3.

Ключи не должны находиться в исходном коде или попадать в GitHub.

Необходимые библиотеки:

- pandas;
- requests;
- python-dotenv;
- kaggle;
- ijson;
- beautifulsoup4.

Установка выполняется один раз в используемом Python-окружении.

In [2]:
packages = [
    "pandas",
    "requests",
    "python-dotenv",
    "kaggle",
    "ijson",
    "beautifulsoup4",
    "kagglehub"
]

subprocess.check_call([
    sys.executable, "-m", "pip", "install", *packages
])

0

## 2. Загрузка реляционных наборов данных

Для описания характеристик видеоигр и факторов интереса аудитории
используются три открытых набора данных:

1. FronkonGames Steam Games Dataset — сведения об играх Steam.
2. RAWG Games Dataset — дополнительная информация об играх.
3. Top 1000 Twitch Streamers — характеристики стримеров и показатели
   зрительской активности.

Источники загружаются из Kaggle в исходном виде.
Файлы сохраняются в отдельных каталогах `raw/`.

Это позволяет повторить сбор данных на другом устройстве
без предварительного копирования подготовленных таблиц.

In [3]:
datasets = {
    "steam_games_fronkongames": {
        "handle": "fronkongames/steam-games-dataset",
        "raw_dir": "steam/fronkongames",
        "description": "Steam games dataset; games.json is the required source file",
    },
    
    "rawg_games": {
        "handle": "atalaydenknalbant/rawg-games-dataset",
        "raw_dir": "rawg",
        "description": "RAWG games dataset",
    },
    
    "twitch_streamers": {
        "handle": "hibrahimag1/top-1000-twitch-streamers-data-may-2024",
        "raw_dir": "twitch/streamers",
        "description": "Top 1000 Twitch streamers, May 2024",
    },
    
    "steam_recommendations": {
        "handle": "antonkozyriev/game-recommendations-on-steam",
        "raw_dir": "steam/recommendations",
        "description": "Steam game recommendations",
    },

    "twitch_evolution": {
        "handle": "rankirsh/evolution-of-top-games-on-twitch",
        "raw_dir": "twitch/evolution",
        "description": "Evolution of top games on Twitch",
    },

    "steam_monthly_players": {
        "handle": "lunthu/steam-monthly-average-players",
        "raw_dir": "steam/monthly_players",
        "description": "Steam monthly average players",
    }
}

#Загрузка

def download_kaggle_dataset(info):
    try:
        return Path(kagglehub.dataset_download(info["handle"]))
    except Exception as e:
        print(f"Ошибка загрузки {info['handle']}: {e}")
        return None


def copy_dataset_to_raw(source, destination):
    destination.mkdir(parents=True, exist_ok=True)

    for file in source.rglob("*"):
        if not file.is_file():
            continue

        target = destination / file.relative_to(source)
        target.parent.mkdir(parents=True, exist_ok=True)

        if target.exists() and target.stat().st_size == file.stat().st_size:
            continue

        shutil.copy2(file, target)

    return destination


downloaded_datasets = {}

for name, info in datasets.items():
    print(f"\nDownloading: {info['handle']}")

    source = download_kaggle_dataset(info)

    if source is not None:
        target = copy_dataset_to_raw(
            source,
            RAW / info["raw_dir"],
        )

        downloaded_datasets[name] = {
            "handle": info["handle"],
            "kaggle_path": source,
            "raw_path": target,
        }

        print(f"Saved to: {target}")


Downloading: fronkongames/steam-games-dataset
Resuming download from 14680064 bytes (312837808 bytes left)...
Resuming download to C:\Users\Daniil\.cache\kagglehub\datasets\fronkongames\steam-games-dataset\58.archive (14680064/327517872) bytes left.


100%|██████████| 312M/312M [00:12<00:00, 24.9MB/s] 

Extracting files...


Saved to: C:\Users\Daniil\Documents\GitHub\gaming-data-analysis\raw\steam\fronkongames

Downloading: atalaydenknalbant/rawg-games-dataset


100%|██████████| 1.92G/1.92G [01:23<00:00, 24.7MB/s]

Extracting files...


... (8 outputs left out)

Saved to: C:\Users\Daniil\Documents\GitHub\gaming-data-analysis\raw\twitch\evolution

Downloading: lunthu/steam-monthly-average-players


100%|██████████| 8.28M/8.28M [00:00<00:00, 10.5MB/s]

Extracting files...


Saved to: C:\Users\Daniil\Documents\GitHub\gaming-data-analysis\raw\steam\monthly_players


In [4]:
files = []

for name, info in downloaded_datasets.items():
    for path in info["raw_path"].rglob("*"):
        if path.is_file():
            files.append({
                "dataset": name,
                "source": info["handle"],
                "filename": path.name,
                "relative_path": str(path.relative_to(ROOT)),
                "extension": path.suffix.lower(),
                "size_mb": round(path.stat().st_size / 1024**2, 2),
            })

inventory = pd.DataFrame(files)

display(inventory.sort_values("size_mb", ascending=False))

print(f"Датасетов загружено: {len(downloaded_datasets)}")
print(f"Файлов найдено: {len(inventory)}")
print(f"Общий размер: {inventory['size_mb'].sum() / 1024:.2f} GB")

,dataset,source,filename,relative_path,extension,size_mb
4,rawg_games,atalaydenknalbant/rawg-games-dataset,videogames_data.jsonl,raw\rawg\jsonl\videogames_data.jsonl,.jsonl,4101.47
8,steam_recommendations,antonkozyriev/game-recommendations-on-steam,recommendations.csv,raw\steam\recommendations\recommendations.csv,.csv,1929.50
2,rawg_games,atalaydenknalbant/rawg-games-dataset,videogames_data.csv,raw\rawg\videogames_data.csv,.csv,1907.80
3,rawg_games,atalaydenknalbant/rawg-games-dataset,videogames_data.parquet,raw\rawg\videogames_data.parquet,.parquet,964.80
1,steam_games_fronkongames,fronkongames/steam-games-dataset,games.json,raw\steam\fronkongames\games.json,.json,960.92
0,steam_games_fronkongames,fronkongames/steam-games-dataset,games.csv,raw\steam\fronkongames\games.csv,.csv,382.56
9,steam_recommendations,antonkozyriev/game-recommendations-on-steam,users.csv,raw\steam\recommendations\users.csv,.csv,184.12
12,steam_monthly_players,lunthu/steam-monthly-average-players,steamcharts.csv,raw\steam\monthly_players\steamcharts.csv,.csv,36.13
7,steam_recommendations,antonkozyriev/game-recommendations-on-steam,games_metadata.json,raw\steam\recommendations\games_metadata.json,.json,16.74
6,steam_recommendations,antonkozyriev/game-recommendations-on-steam,games.csv,raw\steam\recommendations\games.csv,.csv,4.64


Датасетов загружено: 6
Файлов найдено: 13
Общий размер: 10.24 GB


## 3. Получение данных YouTube Data API v3

YouTube используется как дополнительный источник информации
об интересе аудитории к видеоиграм.

Для ограничения объёма запросов выбираются 30 игр с наибольшим
количеством рекомендаций в Steam.

Для каждой игры выполняется поиск видеороликов.
Сохраняются доступные сведения о видео:

- название и описание;
- дата публикации;
- название канала;
- количество просмотров;
- количество отметок «Нравится»;
- количество комментариев.

Данные YouTube не являются прямой статистикой продаж или количества
игроков. Они характеризуют отдельный аспект интереса аудитории —
внимание к игровому контенту на видеоплатформе.

Результаты сохраняются в JSON, чтобы не терять исходную структуру
ответов API.

In [6]:
import ijson
import heapq

steam_json = next((RAW / "steam").rglob("games.json"), None)

if steam_json is None:
    raise FileNotFoundError("Не найден games.json в raw/steam")

top_games = []

with open(steam_json, "rb") as file:
    for appid, game in ijson.kvitems(file, ""):
        count = game.get("recommendations")

        if game.get("name") and isinstance(count, (int, float)):
            top_games.append((
                int(count),
                str(appid),
                game["name"],
            ))

top_games = heapq.nlargest(30, top_games)

top30 = pd.DataFrame(
    top_games,
    columns=["recommendations", "appid", "game_name"],
)

display(top30)

,recommendations,appid,game_name
0,4830455,730,Counter-Strike 2
1,1854262,271590,Grand Theft Auto V Legacy
2,1753261,578080,PUBG: BATTLEGROUNDS
3,1224102,488824,Tom Clancy's Rainbow Six® Siege X
4,1223854,359550,Tom Clancy's Rainbow Six® Siege X
5,1223842,488823,Tom Clancy's Rainbow Six® Siege X
6,1223680,488821,Tom Clancy's Rainbow Six® Siege X
7,1223678,488822,Tom Clancy's Rainbow Six® Siege X
8,1163012,105600,Terraria
9,1081623,252490,Rust


In [8]:
api_key = os.getenv("YOUTUBE_API_KEY")

if not api_key:
    raise ValueError("Добавь YOUTUBE_API_KEY в файл .env")

youtube_url = "https://www.googleapis.com/youtube/v3/search"

search_results = []

for game_name in top30["game_name"]:
    response = requests.get(
        youtube_url,
        params={
            "part": "snippet",
            "q": f"{game_name} game",
            "type": "video",
            "maxResults": 10,
            "key": api_key,
        },
        timeout=30,
    )
    response.raise_for_status()

    data = response.json()

    for item in data.get("items", []):
        video_id = item.get("id", {}).get("videoId")

        if not video_id:
            continue

        snippet = item.get("snippet", {})

        search_results.append({
            "game_name": game_name,
            "video_id": video_id,
            "title": snippet.get("title"),
            "description": snippet.get("description"),
            "channel_title": snippet.get("channelTitle"),
            "published_at": snippet.get("publishedAt"),
        })

print("Найдено видео:", len(search_results))

Найдено видео: 300


In [9]:
video_ids = list({
    item["video_id"] for item in search_results
})

video_statistics = {}

# YouTube videos.list принимает до 50 ID за один запрос
for start in range(0, len(video_ids), 50):
    batch = video_ids[start:start + 50]

    response = requests.get(
        videos_url,
        params={
            "part": "statistics",
            "id": ",".join(batch),
            "key": api_key,
        },
        timeout=30,
    )
    response.raise_for_status()

    for item in response.json().get("items", []):
        video_statistics[item["id"]] = item.get("statistics", {})

for item in search_results:
    stats = video_statistics.get(item["video_id"], {})

    item["views"] = stats.get("viewCount")
    item["likes"] = stats.get("likeCount")
    item["comments"] = stats.get("commentCount")

youtube_dir = RAW / "youtube"

with open(
    youtube_dir / "youtube_search_results.json",
    "w",
    encoding="utf-8",
) as file:
    json.dump(search_results, file, ensure_ascii=False, indent=2)

with open(
    youtube_dir / "youtube_video_statistics.json",
    "w",
    encoding="utf-8",
) as file:
    json.dump(video_statistics, file, ensure_ascii=False, indent=2)

youtube_df = pd.DataFrame(search_results)

youtube_df.to_csv(
    youtube_dir / "youtube_videos.csv",
    index=False,
    encoding="utf-8-sig",
)

display(youtube_df.head())
print("Сохранено видео:", len(youtube_df))

,game_name,video_id,title,description,channel_title,published_at,views,likes,comments
0,Counter-Strike 2,QVctog5F_jY,Counter Strike 2 Ranked Gameplay 4K (No Commen...,Counter Strike 2 Ranked Gameplay 4K (No Commen...,Hytron,2024-04-24T13:00:32Z,2334608,19767,437
1,Counter-Strike 2,nSE38xjMLqE,Counter-Strike 2 - Launch Trailer,Counter-Strike 2 is here! A free upgrade to CS...,Valve,2023-09-27T20:43:52Z,2761445,129849,12734
2,Counter-Strike 2,x-m24USPutM,Counter Strike 2 Gameplay 4K (No Commentary),Counter Strike 2 Ranked Gameplay 4K (No Commen...,Hytron,2023-11-30T13:00:49Z,93184,756,16
3,Counter-Strike 2,foeJldpwZLc,NEW TOP OF THE DOOR LINEUP ON DUST 2 #shorts #cs2,,Gold Cs Nades,2026-11-09T11:00:12Z,16703,714,7
4,Counter-Strike 2,P22HqM9w500,CS2- 33 Kills On Dust 2 Competitive Full Gamep...,CS2- 33 Kills On Dust 2 Competitive Full Gamep...,Kryptek Muted,2024-01-04T00:12:41Z,1575326,12940,448


Сохранено видео: 300


## 4. Сбор неструктурированных текстовых данных

Для дополнительного изучения игровой аудитории и рынка используются
две публикации Habr.

Тексты статей представляют собой неструктурированные данные.
В дальнейшем они будут использованы для демонстрации методов
обработки естественного языка:

- извлечение и очистка текста;
- токенизация;
- лемматизация;
- аннотирование;
- TF-IDF-векторизация.

Выбранные публикации:

1. «Только 12,5% игр в „Стиме“ получают пользовательский отклик»;
2. «От ниши к мейнстриму: с 33 до 75% за 5 лет. Как игры захватили Россию».

Две статьи подходят для демонстрации процесса обработки текста,
но не образуют репрезентативную выборку для самостоятельных
статистических выводов о всей игровой индустрии.

In [11]:
from bs4 import BeautifulSoup

habr_articles = [
    {
        "url": "https://habr.com/ru/articles/995396/",
        "title": "Только 12,5% игр в Steam получают пользовательский отклик",
    },
    {
    "url": "https://habr.com/ru/articles/990554/",
    "title": "От ниши к мейнстриму: с 33 до 75% за 5 лет. Как игры захватили Россию",
    }
]

habr_results = []

for article in habr_articles:
    response = requests.get(
        article["url"],
        headers={"User-Agent": "Mozilla/5.0"},
        timeout=30,
    )

    result = {
        **article,
        "status_code": response.status_code,
        "text": None,
    }

    if response.ok:
        soup = BeautifulSoup(response.text, "html.parser")

        content = soup.select_one(
            ".article-formatted-body, .tm-article-body"
        )

        if content:
            result["text"] = content.get_text(" ", strip=True)

    habr_results.append(result)

habr_dir = RAW / "habr"

with open(
    habr_dir / "articles.json",
    "w",
    encoding="utf-8",
) as file:
    json.dump(habr_results, file, ensure_ascii=False, indent=2)

for article in habr_results:
    print(
        article["title"],
        "| HTTP:", article["status_code"],
        "| Текст получен:", bool(article["text"]),
    )

Только 12,5% игр в Steam получают пользовательский отклик | HTTP: 200 | Текст получен: True
От ниши к мейнстриму: с 33 до 75% за 5 лет. Как игры захватили Россию | HTTP: 200 | Текст получен: True


## 5. Иерархические данные в формате XML

Методические указания требуют включить в исследование хотя бы
один иерархический набор данных в формате XML.

В качестве планируемого источника рассматривается Росстат.
Предполагается использовать социально-экономические показатели,
которые могут характеризовать внешние факторы игровой индустрии.

На момент выполнения данного этапа конкретная XML-выгрузка
ещё не добавлена в проект.

Поэтому требование о наличии XML-источника пока имеет статус
«не выполнено». После получения файла необходимо сохранить
оригинал в `raw/rosstat/` и описать его структуру.

In [13]:
import ijson
import pyarrow.parquet as pq

for source in ["steam", "rawg", "twitch"]:
    print(f"\n{'=' * 60}\n{source.upper()}")

    source_files = [
        p for p in (RAW / source).rglob("*")
        if p.is_file() and p.suffix.lower() in [".csv", ".parquet", ".json", ".jsonl"]
    ]

    if not source_files:
        print("Файлы не найдены")
        continue

    for path in source_files:
        print(f"\nФайл: {path.name}")

        try:
            if path.suffix.lower() == ".csv":
                sample = pd.read_csv(path, nrows=5)

            elif path.suffix.lower() == ".parquet":
                parquet_file = pq.ParquetFile(path)
                sample = next(parquet_file.iter_batches(batch_size=5)).to_pandas()

            # Отдельная логика под жсоны тк у них другие уровни вложенности
            elif path.name == "games.json" and source == "steam":
                records = []

                with open(path, "rb") as file:
                    for appid, game in ijson.kvitems(file, ""):
                        records.append({"AppID": appid, **game})
                        if len(records) == 5:
                            break

                sample = pd.DataFrame(records)

            elif path.suffix.lower() == ".jsonl":
                sample = pd.read_json(path, lines=True, nrows=5)

            else:
                try:
                    sample = pd.read_json(path).head(5)
                except ValueError:
                    sample = pd.read_json(path, lines=True, nrows=5)

            print("Столбцы:", list(sample.columns))
            display(sample)

        except Exception as error:
            print("Не удалось прочитать файл:", error)


STEAM

Файл: games.csv
Столбцы: ['AppID', 'Name', 'Release date', 'Estimated owners', 'Peak CCU', 'Required age', 'Price', 'DiscountDLC count', 'About the game', 'Supported languages', 'Full audio languages', 'Reviews', 'Header image', 'Website', 'Support url', 'Support email', 'Windows', 'Mac', 'Linux', 'Metacritic score', 'Metacritic url', 'User score', 'Positive', 'Negative', 'Score rank', 'Achievements', 'Recommendations', 'Notes', 'Average playtime forever', 'Average playtime two weeks', 'Median playtime forever', 'Median playtime two weeks', 'Developers', 'Publishers', 'Categories', 'Genres', 'Tags', 'Screenshots', 'Movies']


,AppID,Name,Release date,Estimated owners,Peak CCU,Required age,Price,DiscountDLC count,About the game,Supported languages,...,Average playtime two weeks,Median playtime forever,Median playtime two weeks,Developers,Publishers,Categories,Genres,Tags,Screenshots,Movies
2539430,Black Dragon Mage Playtest,"Aug 1, 2023",0 - 0,0,0,0.00,0,0,NaN,[],...,0,0,0,NaN,NaN,NaN,NaN,NaN,https://shared.akamai.steamstatic.com/store_it...,NaN
496350,Supipara - Chapter 1 Spring Has Come!,"Jul 29, 2016",0 - 20000,0,0,5.24,65,0,"Springtime, April: when the cherry trees come ...",['English'],...,0,8,0,minori,MangaGamer,"Single-player,Steam Trading Cards,Steam Cloud,...",Adventure,"Adventure,Visual Novel,Anime,Cute",https://shared.akamai.steamstatic.com/store_it...,NaN
1034400,Mystery Solitaire The Black Raven,"May 6, 2019",0 - 20000,0,0,4.99,0,0,"Immerse yourself in the most beloved, mystical...","['English', 'French', 'German', 'Russian']",...,0,0,0,Somer Games,8floor,"Single-player,Family Sharing",Casual,"Casual,Card Game,Solitaire,Puzzle,Hidden Objec...",https://shared.akamai.steamstatic.com/store_it...,NaN
3292190,버튜버 파라노이아 - Vtuber Paranoia,"Oct 31, 2024",0 - 20000,1,0,8.99,0,1,"synopsis 'Hello, I'm Hiyoro, a new YouTuber!' ...",['Korean'],...,0,0,0,유진게임즈,유진게임즈,"Single-player,Steam Achievements,Family Sharing","Casual,Indie,Simulation",NaN,https://shared.akamai.steamstatic.com/store_it...,NaN
3631080,Maze Quest VR,"Apr 24, 2025",0 - 20000,0,0,4.99,0,0,Its not just a Maze; its a Quest! Enter the ca...,['English'],...,0,0,0,Reality Expanded LLC,Reality Expanded LLC,"Single-player,VR Only,Steam Leaderboards,Famil...","Action,Early Access",NaN,https://shared.akamai.steamstatic.com/store_it...,NaN



Файл: games.json
Столбцы: ['AppID', 'name', 'release_date', 'required_age', 'price', 'dlc_count', 'detailed_description', 'about_the_game', 'short_description', 'reviews', 'header_image', 'website', 'support_url', 'support_email', 'windows', 'mac', 'linux', 'metacritic_score', 'metacritic_url', 'achievements', 'recommendations', 'notes', 'supported_languages', 'full_audio_languages', 'packages', 'developers', 'publishers', 'categories', 'genres', 'screenshots', 'movies', 'user_score', 'score_rank', 'positive', 'negative', 'estimated_owners', 'average_playtime_forever', 'average_playtime_2weeks', 'median_playtime_forever', 'median_playtime_2weeks', 'discount', 'peak_ccu', 'tags']


,AppID,name,release_date,required_age,price,dlc_count,detailed_description,about_the_game,short_description,reviews,...,positive,negative,estimated_owners,average_playtime_forever,average_playtime_2weeks,median_playtime_forever,median_playtime_2weeks,discount,peak_ccu,tags
0,2539430,Black Dragon Mage Playtest,"Aug 1, 2023",0,0.0,0,,,,,...,0,0,0 - 0,0,0,0,0,0,0,[]
1,496350,Supipara - Chapter 1 Spring Has Come!,"Jul 29, 2016",0,5.24,0,"Springtime, April: when the cherry trees come ...","Springtime, April: when the cherry trees come ...","Spring has come, and our protagonist, Yukinari...",,...,252,3,0 - 20000,8,0,8,0,65,0,"{'Adventure': 27, 'Visual Novel': 19, 'Anime':..."
2,1034400,Mystery Solitaire The Black Raven,"May 6, 2019",0,4.99,0,"Immerse yourself in the most beloved, mystical...","Immerse yourself in the most beloved, mystical...",Discover an entrancing and spectacular world!,,...,21,3,0 - 20000,0,0,0,0,0,0,"{'Casual': 83, 'Card Game': 52, 'Solitaire': 4..."
3,3292190,버튜버 파라노이아 - Vtuber Paranoia,"Oct 31, 2024",0,8.99,1,"synopsis 'Hello, I'm Hiyoro, a new YouTuber!' ...","synopsis 'Hello, I'm Hiyoro, a new YouTuber!' ...",Yuha! I'll start the broadcast! Hakko's extrem...,,...,0,0,0 - 20000,0,0,0,0,0,1,[]
4,3631080,Maze Quest VR,"Apr 24, 2025",0,4.99,0,Its not just a Maze; its a Quest! Enter the ca...,Its not just a Maze; its a Quest! Enter the ca...,Its not just a Maze; its a Quest! Enter the ca...,,...,0,0,0 - 20000,0,0,0,0,0,0,[]



Файл: steamcharts.csv
Столбцы: ['month', 'avg_players', 'gain', 'gain_percent', 'peak_players', 'name', 'steam_appid']


,month,avg_players,gain,gain_percent,peak_players,name,steam_appid
0,Sep-25,7805.25,883.12,0.1276,13254,Counter-Strike,10
1,Aug-25,6922.13,-449.35,-0.0610,12168,Counter-Strike,10
2,Jul-25,7371.48,-833.50,-0.1016,13951,Counter-Strike,10
3,Jun-25,8204.98,-847.53,-0.0936,15798,Counter-Strike,10
4,May-25,9052.51,-471.31,-0.0495,15333,Counter-Strike,10


... (16 outputs left out)


Файл: Twitch_global_data.csv
Столбцы: ['year', 'Month', 'Hours_watched', 'Avg_viewers', 'Peak_viewers', 'Streams', 'Avg_channels', 'Games_streamed', 'Viewer_ratio']


,year,Month,Hours_watched,Avg_viewers,Peak_viewers,Streams,Avg_channels,Games_streamed,Viewer_ratio
0,2016,1,480241904,646355,1275257,7701675,20076,12149,29.08
1,2016,2,441859897,635769,1308032,7038520,20427,12134,28.98
2,2016,3,490669308,660389,1591551,7390957,20271,12234,28.92
3,2016,4,377975447,525696,1775120,6869719,16791,12282,28.80
4,2016,5,449836631,605432,1438962,7535519,19394,12424,28.85



Файл: datasetV2.csv
Столбцы: ['RANK', 'NAME', 'LANGUAGE', 'TYPE', 'MOST_STREAMED_GAME', '2ND_MOST_STREAMED_GAME', 'AVERAGE_STREAM_DURATION', 'FOLLOWERS_GAINED_PER_STREAM', 'AVG_VIEWERS_PER_STREAM', 'AVG_GAMES_PER_STREAM', 'TOTAL_TIME_STREAMED', 'TOTAL_FOLLOWERS', 'TOTAL_VIEWS', 'TOTAL_GAMES_STREAMED', 'ACTIVE_DAYS_PER_WEEK', 'MOST_ACTIVE_DAY', 'DAY_WITH_MOST_FOLLOWERS_GAINED']


,RANK,NAME,LANGUAGE,TYPE,MOST_STREAMED_GAME,2ND_MOST_STREAMED_GAME,AVERAGE_STREAM_DURATION,FOLLOWERS_GAINED_PER_STREAM,AVG_VIEWERS_PER_STREAM,AVG_GAMES_PER_STREAM,TOTAL_TIME_STREAMED,TOTAL_FOLLOWERS,TOTAL_VIEWS,TOTAL_GAMES_STREAMED,ACTIVE_DAYS_PER_WEEK,MOST_ACTIVE_DAY,DAY_WITH_MOST_FOLLOWERS_GAINED
0,1,kaicenat,English,personality,Just Chatting,I'm Only Sleeping,7.6,18405,15852,2.3,4698,10600000,9150000,194,3.6,Friday,Saturday
1,2,jynxzi,English,personality,Tom Clancy's Rainbow Six Siege,NBA 2K20,5.4,3386,1145,1.2,8407,5760000,1950000,54,5.6,Tuesday,Sunday
2,3,caedrel,English,personality,League of Legends,I'm Only Sleeping,6.3,689,12331,1.3,6728,797000,14200000,111,2.8,Thursday,Sunday
3,4,caseoh_,English,personality,NBA 2K23,Just Chatting,4.6,7185,0,3.6,2554,4220000,53,385,6.2,Friday,Monday
4,5,ibai,Spanish,personality,Just Chatting,League of Legends,4.1,8289,190714,1.5,6865,15600000,359000000,149,4.3,Wednesday,Saturday


## 6. Аналитический портрет объектов исследования

На основании фактического просмотра загруженных наборов данных выделяются следующие объекты исследования и группы характеристик. В аналитический портрет включены атрибуты, присутствующие в схемах файлов. Отдельные характеристики могут отсутствовать у некоторых объектов или иметь пропущенные значения.

### 6.1. Steam

**Объект:** видеоигра, пользовательская оценка игры и игровая активность.

**Категориальные характеристики:**

- название игры;
- жанры;
- категории;
- разработчики;
- издатели;
- поддерживаемые платформы (Windows, macOS, Linux);
- теги;
- поддерживаемые языки.

**Числовые характеристики:**

- цена;
- количество владельцев (оценочный диапазон);
- пиковое количество одновременно играющих пользователей (Peak CCU);
- оценка Metacritic;
- пользовательская оценка;
- количество рекомендаций;
- количество положительных и отрицательных отзывов;
- количество достижений;
- среднее и медианное игровое время;
- количество пользователей, оставивших отзывы;
- количество часов, проведённых пользователями в игре.

**Временные характеристики:**

- дата выхода игры;
- дата публикации пользовательского отзыва;
- месяц и год статистики среднемесячного количества игроков.

Для анализа используются несколько наборов данных Steam: каталог игр FronkonGames, набор пользовательских рекомендаций и отзывов, а также статистика среднемесячного количества игроков SteamCharts.

### 6.2. RAWG

**Объект:** видеоигра.

**Категориальные характеристики:**

- название игры;
- жанры;
- платформы;
- разработчики;
- издатели;
- теги;
- возрастной рейтинг ESRB;
- игровые серии и дополнительные материалы (при наличии).

**Числовые характеристики:**

- пользовательский рейтинг;
- максимальная оценка рейтинговой шкалы;
- количество пользовательских оценок;
- количество текстовых отзывов;
- общее количество отзывов;
- оценка Metacritic;
- игровое время;
- количество добавлений игры пользователями;
- количество рекомендаций;
- количество достижений;
- количество связанных видеороликов и других материалов.

**Временные характеристики:**

- дата выхода игры;
- дата обновления записи в RAWG.

Набор данных содержит как простые поля, так и вложенные структуры, например сведения о платформах, жанрах, тегах и рейтингах.

### 6.3. Twitch

**Объект:** стример, игровая активность и популярность видеоигр на платформе.

**Категориальные характеристики:**

- имя стримера;
- язык канала;
- тип стримера;
- наиболее часто транслируемая игра;
- вторая наиболее часто транслируемая игра;
- наиболее активный день недели.

**Числовые характеристики стримеров:**

- средняя продолжительность трансляции;
- прирост подписчиков за трансляцию;
- среднее количество зрителей;
- количество часов трансляций;
- общее количество подписчиков;
- общее количество просмотров;
- количество транслируемых игр;
- количество активных дней в неделю.

**Числовые характеристики популярности игр:**

- время просмотра (Hours Watched);
- время трансляций (Hours Streamed);
- пиковое количество зрителей;
- пиковое количество каналов;
- количество стримеров;
- среднее количество зрителей;
- среднее количество каналов;
- соотношение времени просмотра к времени трансляций.

**Временные характеристики:**

- месяц и год статистики;
- период активности стримера.

Для анализа используются два направления данных: характеристики отдельных стримеров и агрегированная статистика популярности игр и активности на Twitch.

### 6.4. YouTube

**Объект:** видеоролик, связанный с видеоигрой.

**Категориальные характеристики:**

- название игры, по которой выполнен поиск (`game_name`);
- название канала (`channel_title`);
- название видеоролика (`title`);
- идентификатор видеоролика (`video_id`).

**Числовые характеристики:**

- количество просмотров (`views`);
- количество отметок «Нравится» (`likes`);
- количество комментариев (`comments`).

**Временная характеристика:**

- дата и время публикации видеоролика (`published_at`).

**Дополнительная характеристика:**

- описание видеоролика (`description`), используемое для анализа содержания игрового видеоконтента.

Данные получены с использованием YouTube Data API v3. В результате выполнения запроса сформирована выборка из 300 видеороликов, связанных с исследуемыми видеоиграми.

Основное назначение источника — исследование интереса аудитории к игровому видеоконтенту на основе просмотров, реакций пользователей и временных характеристик публикаций.

### 6.5. Habr

**Объект:** текстовая публикация об игровой индустрии.

**Категориальные характеристики:**

- заголовок публикации;
- автор;
- источник публикации;
- тематическая принадлежность.

**Числовые характеристики:**

- показатели просмотров, комментариев и других реакций, если они присутствуют в полученных данных.

**Временная характеристика:**

- дата публикации.

Основное назначение источника — анализ содержания публикаций и демонстрация методов обработки естественного языка. Основным аналитическим материалом выступает текст статьи.

### Ограничения

Перечисленные характеристики определены по фактическим схемам просмотренных файлов и не означают, что все поля заполнены для каждой записи.

Необходимо учитывать различия в структуре источников, наличие пропусков, вложенных атрибутов и неодинаковые способы измерения показателей.

Характеристики YouTube и отдельные метаданные Habr требуют дополнительной проверки по фактически сохранённым ответам API и страницам публикаций.

Неподтверждённые атрибуты не создаются искусственно и не используются в последующем анализе без проверки.

In [15]:

source_registry = pd.DataFrame([
    {
        "source": "FronkonGames Steam Games Dataset",
        "type": "Табличный / иерархический",
        "format": "CSV / JSON",
        "purpose": "Каталог игр Steam и их характеристики",
        "status": "Загружен",
    },
    {
        "source": "Game Recommendations on Steam — Games",
        "type": "Табличный",
        "format": "CSV",
        "purpose": "Характеристики игр, оценки и цены",
        "status": "Загружен",
    },
    {
        "source": "Game Recommendations on Steam — Recommendations",
        "type": "Табличный",
        "format": "CSV",
        "purpose": "Пользовательские отзывы, рекомендации и игровое время",
        "status": "Загружен",
    },
    {
        "source": "Game Recommendations on Steam — Users",
        "type": "Табличный",
        "format": "CSV",
        "purpose": "Характеристики пользователей Steam",
        "status": "Загружен",
    },
    {
        "source": "Game Recommendations on Steam — Games Metadata",
        "type": "Иерархический",
        "format": "JSON",
        "purpose": "Описания и теги игр",
        "status": "Загружен",
    },
    {
        "source": "Steam Monthly Average Players",
        "type": "Табличный",
        "format": "CSV",
        "purpose": "Динамика среднемесячного количества игроков",
        "status": "Загружен",
    },
    {
        "source": "RAWG Games Dataset",
        "type": "Табличный / иерархический",
        "format": "CSV / Parquet / JSONL",
        "purpose": "Характеристики, рейтинги и метаданные видеоигр",
        "status": "Загружен",
    },
    {
        "source": "Top 1000 Twitch Streamers",
        "type": "Табличный",
        "format": "CSV",
        "purpose": "Характеристики стримеров и их аудитории",
        "status": "Загружен",
    },
    {
        "source": "Evolution of Top Games on Twitch — Game Data",
        "type": "Табличный",
        "format": "CSV",
        "purpose": "Динамика популярности отдельных игр на Twitch",
        "status": "Загружен",
    },
    {
        "source": "Evolution of Top Games on Twitch — Global Data",
        "type": "Табличный",
        "format": "CSV",
        "purpose": "Общая динамика активности пользователей Twitch",
        "status": "Загружен",
    },
    {
    "source": "YouTube Data API v3",
    "type": "API",
    "format": "JSON",
    "purpose": "Интерес аудитории к игровому видеоконтенту",
    "status": "Загружен (300 видеороликов)",
    },
    {
        "source": "Habr",
        "type": "Неструктурированные данные",
        "format": "HTML / текст",
        "purpose": "Текстовый анализ публикаций об игровой индустрии",
        "status": "Загружен (2 статьи)",
    },
    {
        "source": "Росстат",
        "type": "Иерархический",
        "format": "XML",
        "purpose": "Внешние социально-экономические факторы",
        "status": "Ожидает XML-файл",
    },
])

source_registry["collection_date"] = datetime.now(
    timezone.utc
).isoformat()

display(source_registry)

docs_dir = ROOT / "docs"
docs_dir.mkdir(parents=True, exist_ok=True)

source_registry.to_csv(
    docs_dir / "data_sources_registry.csv",
    index=False,
    encoding="utf-8-sig",
)


,source,type,format,purpose,status,collection_date
0,FronkonGames Steam Games Dataset,Табличный / иерархический,CSV / JSON,Каталог игр Steam и их характеристики,Загружен,2026-11-09T18:58:37.661797+00:00
1,Game Recommendations on Steam — Games,Табличный,CSV,"Характеристики игр, оценки и цены",Загружен,2026-11-09T18:58:37.661797+00:00
2,Game Recommendations on Steam — Recommendations,Табличный,CSV,"Пользовательские отзывы, рекомендации и игрово...",Загружен,2026-11-09T18:58:37.661797+00:00
3,Game Recommendations on Steam — Users,Табличный,CSV,Характеристики пользователей Steam,Загружен,2026-11-09T18:58:37.661797+00:00
4,Game Recommendations on Steam — Games Metadata,Иерархический,JSON,Описания и теги игр,Загружен,2026-11-09T18:58:37.661797+00:00
5,Steam Monthly Average Players,Табличный,CSV,Динамика среднемесячного количества игроков,Загружен,2026-11-09T18:58:37.661797+00:00
6,RAWG Games Dataset,Табличный / иерархический,CSV / Parquet / JSONL,"Характеристики, рейтинги и метаданные видеоигр",Загружен,2026-11-09T18:58:37.661797+00:00
7,Top 1000 Twitch Streamers,Табличный,CSV,Характеристики стримеров и их аудитории,Загружен,2026-11-09T18:58:37.661797+00:00
8,Evolution of Top Games on Twitch — Game Data,Табличный,CSV,Динамика популярности отдельных игр на Twitch,Загружен,2026-11-09T18:58:37.661797+00:00
9,Evolution of Top Games on Twitch — Global Data,Табличный,CSV,Общая динамика активности пользователей Twitch,Загружен,2026-11-09T18:58:37.661797+00:00


## 7. Выводы по результатам подбора источников

В ходе этапа сформирована эмпирическая база исследования,
включающая сведения об играх, активности стримеров, видеоконтенте
и публикациях об игровой индустрии.

Выбранные источники позволяют рассматривать популярность
видеоигр с нескольких сторон:

- характеристики и свойства самих игр;
- пользовательские оценки и рекомендации;
- зрительский интерес к игровому контенту;
- тематические и содержательные характеристики публикаций.

Исходные данные сохранены в каталоге `raw/`.
Сведения об источниках и результатах загрузки отражены
в реестре `docs/data_sources_registry.csv`.

На момент выполнения этапа остаётся незакрытым требование
о наличии иерархического XML-источника. Для его выполнения
необходимо добавить реальную XML-выгрузку Росстата
или другой релевантный XML-набор данных.

Далее выполняется этап 2.3 — подготовка данных:
профилирование, проверка пропусков и дубликатов, анализ типов,
очистка, аннотирование текстов и подготовка источников
к последующему объединению.